In [1]:
from __future__ import annotations

from pathlib import Path

from finance_data.bsc import (
    ANALYTIC_METHOD,
    GARCH_MLE_METHOD,
    GARCH_ORACLE_METHOD,
    default_config,
    estimate_main_bundle_runtime,
    run_main_bundle,
    run_oracle_bundle,
)

PROJECT_ROOT = Path.cwd().resolve()
print(f"PROJECT_ROOT = {PROJECT_ROOT}")

PROJECT_ROOT = /Users/rafaeldecara/Programming/Python/rafsbsc/notebooks


## Editable config

In [2]:
cfg = default_config(
    R=5000,
    R_garch=5000,
    dgps=("iid_normal", "garch11_t"),
    methods=(ANALYTIC_METHOD, GARCH_MLE_METHOD),
    n_grid=(60,400,800,1600),
    S_grid=(0.25, 0.5, 0.75),
    max_workers=4,
    seed = 0
)

cfg

ci_levels = (0.90, 0.95, 0.975, 0.99)

eta = estimate_main_bundle_runtime(
    cfg,
    ci_levels=ci_levels,
    scope="main",
    output_dir=PROJECT_ROOT / "outputs" / "bsc_final",
)
print(
    f"ETA: {eta.eta_seconds/60.0:.1f}m "
    f"(range {eta.eta_low_seconds/60.0:.1f}m - {eta.eta_high_seconds/60.0:.1f}m)"
)
print(f"notes={eta.notes}")

ETA: 12.6m (range 10.8m - 13.8m)
notes=Pilot sampled 4/24 cells with R=250, R_garch=250; workers adjustment uses max_workers=4, observed_speedup=1.90 (probe_serial=11.60s, probe_parallel=6.10s, probe_workers=4).


## Simulation

In [3]:
main_bundle = run_main_bundle(cfg, ci_levels=ci_levels, scope="main", progress=True)
results = main_bundle.results
diagnostics = main_bundle.diagnostics
ci_sweep = main_bundle.ci_sweep
main_meta = main_bundle.cache_meta
print(f"[main] cache_hit={main_meta['cache_hit']} hash={main_meta['cache_hash']}")

main bundle:   0%|          | 0/2 [00:00<?, ?it/s]

compute cells:   0%|          | 0/24 [00:00<?, ?it/s]

plot steps:   0%|          | 0/5 [00:00<?, ?it/s]

[main] cache_hit=True hash=4c85c6b268a1


In [4]:
figs = main_bundle.figures

print("Figure keys:", sorted(figs.keys()))
results.head(4)
# display(diagnostics)

Figure keys: ['bias', 'coverage_95', 'coverage_95_vs_n', 'reject_rate_H0_S_eq_0', 'rmse']


,dgp,method,n,S_true,outer_reps,coverage_95,mc_se,mc_lo,mc_hi,reject_rate_H0_S_eq_0,se_cell,omega_hat_cell,se_ratio_meanSE_over_mcSD,fit_fail_count,fit_fail_rate,regularized_count,regularized_rate,fit_converged_rate,fit_attempts_mean,fit_time_ms_mean
0,garch11_t,garch11_mle_analytic,60,0.25,5000,0.9362,0.003456,0.929426,0.942974,0.5002,0.145383,1.753963,1.023315,0,0.0,2289.0,0.4578,1.0,1.0012,15.373255
1,garch11_t,iid_normal_analytic,60,0.25,5000,0.9326,0.003546,0.925651,0.939549,0.5264,0.131999,1.045817,0.929104,0,0.0,NaN,NaN,NaN,NaN,NaN
2,garch11_t,garch11_mle_analytic,60,0.50,5000,0.9108,0.004031,0.902899,0.918701,0.9182,0.170550,3.882415,1.022005,0,0.0,2201.0,0.4402,1.0,1.0004,15.293032
3,garch11_t,iid_normal_analytic,60,0.50,5000,0.9024,0.004197,0.894174,0.910626,0.9538,0.138681,1.155754,0.831034,0,0.0,NaN,NaN,NaN,NaN,NaN


In [5]:
# CI sweep tables from the same config-driven run
avg_ci_length_table = (
    ci_sweep
    .pivot_table(index=["dgp", "method", "n"], columns=["S_true", "ci_level"], values="avg_ci_length")
    .sort_index()
    .sort_index(axis=1)
)

actual_coverage_table = (
    ci_sweep
    .pivot_table(index=["dgp", "method", "n"], columns=["S_true", "ci_level"], values="coverage")
    .sort_index()
    .sort_index(axis=1)
)

print("Average CI length table")
display(avg_ci_length_table)

print("Actual coverage table")
display(actual_coverage_table)


Average CI length table


S_true                                    0.25                                \
ci_level                                 0.900     0.950     0.975     0.990   
dgp        method               n                                              
garch11_t  garch11_mle_analytic 60    0.478269  0.569892  0.651725  0.748965   
                                400   0.193039  0.230021  0.263050  0.302298   
                                800   0.135100  0.160981  0.184097  0.211565   
                                1600  0.094010  0.112019  0.128105  0.147218   
           iid_normal_analytic  60    0.434237  0.517425  0.591724  0.680012   
                                400   0.167247  0.199287  0.227903  0.261907   
                                800   0.118185  0.140827  0.161049  0.185078   
                                1600  0.083540  0.099544  0.113838  0.130823   
iid_normal garch11_mle_analytic 60    0.443183  0.528085  0.603915  0.694021   
                                400   0.167529  0.199623  0.228288  0.262349   
                                800   0.118388  0.141068  0.161324  0.185395   
                                1600  0.083658  0.099685  0.113999  0.131008   
           iid_normal_analytic  60    0.433244  0.516243  0.590372  0.678458   
                                400   0.167142  0.199161  0.227760  0.261742   
                                800   0.118163  0.140800  0.161018  0.185043   
                                1600  0.083530  0.099532  0.113824  0.130807   

S_true                                    0.50                                \
ci_level                                 0.900     0.950     0.975     0.990   
dgp        method               n                                              
garch11_t  garch11_mle_analytic 60    0.561059  0.668543  0.764542  0.878615   
                                400   0.251048  0.299143  0.342098  0.393140   
                                800   0.175393  0.208994  0.239004  0.274665   
                                1600  0.121577  0.144868  0.165670  0.190388   
           iid_normal_analytic  60    0.456220  0.543620  0.621681  0.714438   
                                400   0.174895  0.208400  0.238325  0.273884   
                                800   0.123533  0.147199  0.168336  0.193452   
                                1600  0.087292  0.104015  0.118951  0.136699   
iid_normal garch11_mle_analytic 60    0.478109  0.569702  0.651508  0.748715   
                                400   0.175912  0.209612  0.239711  0.275477   
                                800   0.124249  0.148052  0.169311  0.194573   
                                1600  0.087727  0.104534  0.119544  0.137380   
           iid_normal_analytic  60    0.452706  0.539433  0.616892  0.708935   
                                400   0.174602  0.208051  0.237925  0.273425   
                                800   0.123416  0.147060  0.168176  0.193269   
                                1600  0.087253  0.103968  0.118897  0.136637   

S_true                                    0.75                                
ci_level                                 0.900     0.950     0.975     0.990  
dgp        method               n                                             
garch11_t  garch11_mle_analytic 60    0.662527  0.789449  0.902809  1.037512  
                                400   0.294304  0.350684  0.401040  0.460877  
                                800   0.224181  0.267129  0.305487  0.351067  
                                1600  0.156398  0.186360  0.213120  0.244919  
           iid_normal_analytic  60    0.490674  0.584674  0.668630  0.768392  
                                400   0.186846  0.222641  0.254611  0.292600  
                                800   0.131948  0.157226  0.179803  0.206630  
                                1600  0.093197  0.111051  0.126998  0.145946  
iid_normal garch11_mle_analytic 60    0.516737  0.615730  0.704146  0.809207  
          

Actual coverage table


S_true                                  0.25                            0.50  \
ci_level                               0.900   0.950   0.975   0.990   0.900   
dgp        method               n                                              
garch11_t  garch11_mle_analytic 60    0.8746  0.9362  0.9668  0.9856  0.8440   
                                400   0.9030  0.9514  0.9736  0.9866  0.8906   
                                800   0.9088  0.9582  0.9792  0.9906  0.8928   
                                1600  0.9100  0.9552  0.9768  0.9898  0.9024   
           iid_normal_analytic  60    0.8696  0.9326  0.9652  0.9838  0.8274   
                                400   0.8676  0.9302  0.9632  0.9814  0.7890   
                                800   0.8666  0.9300  0.9616  0.9822  0.7762   
                                1600  0.8642  0.9274  0.9604  0.9806  0.7812   
iid_normal garch11_mle_analytic 60    0.9084  0.9514  0.9746  0.9902  0.9010   
                                400   0.8990  0.9490  0.9732  0.9880  0.9024   
                                800   0.8914  0.9432  0.9706  0.9880  0.9056   
                                1600  0.8986  0.9546  0.9782  0.9900  0.8976   
           iid_normal_analytic  60    0.9062  0.9496  0.9738  0.9900  0.8952   
                                400   0.8986  0.9490  0.9728  0.9878  0.9012   
                                800   0.8912  0.9426  0.9704  0.9880  0.9030   
                                1600  0.8982  0.9538  0.9780  0.9900  0.8952   

S_true                                                          0.75          \
ci_level                               0.950   0.975   0.990   0.900   0.950   
dgp        method               n                                              
garch11_t  garch11_mle_analytic 60    0.9108  0.9482  0.9752  0.7938  0.8682   
                                400   0.9354  0.9628  0.9818  0.8494  0.9082   
                                800   0.9442  0.9696  0.9860  0.8896  0.9360   
                                1600  0.9476  0.9734  0.9894  0.9012  0.9480   
           iid_normal_analytic  60    0.9024  0.9412  0.9722  0.7668  0.8516   
                                400   0.8674  0.9156  0.9524  0.7176  0.7990   
                                800   0.8456  0.9004  0.9406  0.7008  0.7844   
                                1600  0.8554  0.9046  0.9442  0.6890  0.7844   
iid_normal garch11_mle_analytic 60    0.9532  0.9766  0.9926  0.9012  0.9504   
                                400   0.9542  0.9768  0.9910  0.9020  0.9482   
                                800   0.9536  0.9774  0.9900  0.9076  0.9530   
                                1600  0.9512  0.9744  0.9906  0.9008  0.9494   
           iid_normal_analytic  60    0.9504  0.9746  0.9920  0.8962  0.9490   
                                400   0.9516  0.9768  0.9906  0.8988  0.9454   
                                800   0.9530  0.9772  0.9898  0.9050  0.9504   
                                1600  0.9494  0.9734  0.9902  0.8966  0.9480   

S_true                                                
ci_level                               0.975   0.990  
dgp        method               n                     
garch11_t  garch11_mle_analytic 60    0.9128  0.9512  
                                400   0.9404  0.9626  
                                800   0.9628  0.9842  
                                1600  0.9714  0.9864  
           iid_normal_analytic  60    0.9012  0.9448  
                                400   0.8492  0.8982  
                                800   0.8394  0.8952  
                                1600  0.8416  0.8912  
iid_normal garch11_mle_analytic 60    0.9774  0.9898  
                                400   0.9724  0.9884  
                                800   0.9792  0.9922  
                                1600  0.9742  0.9898  
           iid_normal_analytic  60    0.9758  0.9886  
                                400   0.9716  0.9874  
                       

In [6]:
# Sample-Sharpe comparison table (pivoted like CI tables)
import numpy as np
import pandas as pd

required_results = {"dgp", "method", "n", "S_true", "se_cell"}
required_diag = {"dgp", "n", "S_true", "bias"}
missing_results = sorted(required_results - set(results.columns))
missing_diag = sorted(required_diag - set(diagnostics.columns))
if missing_results or missing_diag:
    raise ValueError(
        f"Missing columns. results: {missing_results}; diagnostics: {missing_diag}"
    )

base = diagnostics[["dgp", "n", "S_true", "bias"]].copy()
base["T"] = pd.to_numeric(base["n"], errors="coerce")
if base["T"].isna().any() or (base["T"] <= 1).any():
    raise ValueError("All T values must be numeric and > 1.")

# Runtime bias uses Sharpe with sample sd (ddof=1). Convert to your variance-T definition.
base["S_hat"] = (base["S_true"] + base["bias"]) * np.sqrt(base["T"] / (base["T"] - 1.0))
base["S_hat_minus_S"] = base["S_hat"] - base["S_true"]

summary = results[["dgp", "method", "n", "S_true", "se_cell"]].copy()
summary = summary.merge(
    base[["dgp", "n", "S_true", "T", "S_hat", "S_hat_minus_S"]],
    on=["dgp", "n", "S_true"],
    how="left",
    validate="many_to_one",
)
summary = summary.rename(columns={"se_cell": "standard_error"})

lhs = summary["S_hat_minus_S"].to_numpy(dtype=float)
rhs = (summary["S_hat"] - summary["S_true"]).to_numpy(dtype=float)
mask = np.isfinite(lhs) & np.isfinite(rhs)
if np.any(mask) and not np.allclose(lhs[mask], rhs[mask], rtol=0.0, atol=1e-12):
    raise AssertionError("Identity check failed: S_hat_minus_S != S_hat - S_true")

stat_cols = [
    "S_hat",
    "S_hat_minus_S",
    "standard_error",
]

summary_long = summary[["dgp", "method", "T", "S_true", *stat_cols]].melt(
    id_vars=["dgp", "method", "T", "S_true"],
    value_vars=stat_cols,
    var_name="statistic",
    value_name="value",
)

sample_sharpe_table = (
    summary_long
    .pivot_table(index=["dgp", "method", "T"], columns=["S_true", "statistic"], values="value")
    .sort_index()
    .sort_index(axis=1)
)

present_stats = set(sample_sharpe_table.columns.get_level_values("statistic"))
missing_stats = [s for s in stat_cols if s not in present_stats]
if missing_stats:
    raise AssertionError(f"Missing expected statistics in table columns: {missing_stats}")

print("Sample Sharpe comparison table")
display(sample_sharpe_table)



Sample Sharpe comparison table


S_true                                    0.25                               \
statistic                                S_hat S_hat_minus_S standard_error   
dgp        method               T                                             
garch11_t  garch11_mle_analytic 60    0.269565      0.019565       0.145383   
                                400   0.254836      0.004836       0.058680   
                                800   0.252298      0.002298       0.041067   
                                1600  0.250774      0.000774       0.028577   
           iid_normal_analytic  60    0.269565      0.019565       0.131999   
                                400   0.254836      0.004836       0.050839   
                                800   0.252298      0.002298       0.035926   
                                1600  0.250774      0.000774       0.025394   
iid_normal garch11_mle_analytic 60    0.256138      0.006138       0.134718   
                                400   0.250501      0.000501       0.050925   
                                800   0.251108      0.001108       0.035987   
                                1600  0.250006      0.000006       0.025430   
           iid_normal_analytic  60    0.256138      0.006138       0.131697   
                                400   0.250501      0.000501       0.050807   
                                800   0.251108      0.001108       0.035919   
                                1600  0.250006      0.000006       0.025391   

S_true                                    0.50                               \
statistic                                S_hat S_hat_minus_S standard_error   
dgp        method               T                                             
garch11_t  garch11_mle_analytic 60    0.537096      0.037096       0.170550   
                                400   0.507480      0.007480       0.076313   
                                800   0.504183      0.004183       0.053316   
                                1600  0.502156      0.002156       0.036957   
           iid_normal_analytic  60    0.537096      0.037096       0.138681   
                                400   0.507480      0.007480       0.053164   
                                800   0.504183      0.004183       0.037551   
                                1600  0.502156      0.002156       0.026535   
iid_normal garch11_mle_analytic 60    0.509694      0.009694       0.145335   
                                400   0.501694      0.001694       0.053473   
                                800   0.500989      0.000989       0.037769   
                                1600  0.500617      0.000617       0.026667   
           iid_normal_analytic  60    0.509694      0.009694       0.137613   
                                400   0.501694      0.001694       0.053075   
                                800   0.500989      0.000989       0.037516   
                                1600  0.500617      0.000617       0.026523   

S_true                                    0.75                               
statistic                                S_hat S_hat_minus_S standard_error  
dgp        method               T                                            
garch11_t  garch11_mle_analytic 60    0.805683      0.055683       0.201394  
                                400   0.758990      0.008990       0.089462  
                                800   0.755999      0.005999       0.068146  
                                1600  0.752994      0.002994       0.047542  
           iid_normal_analytic  60    0.805683      0.055683       0.149154  
                                400   0.758990      0.008990       0.056797  
                                800   0.755999      0.005999       0.040109  
                                1600  0.752994      0.002994       0.028330  
iid_normal garch11_mle_analytic 60    0.767531      0.017531       0.157077  
                                400   0.752185      0.002185

In [7]:
# Separate oracle-only config for thesis figures
oracle_s_grid = (
    -1.4,
    -1.2,
    -1.0,
    -0.8,
    -0.6,
    -0.4,
    -0.2,
    0.0,
    0.2,
    0.4,
    0.6,
    0.8,
    1.0,
    1.2,
    1.4,
)
oracle_main_n_grid = (36, 60, 120, 360, 600)
oracle_appendix_n_grid = (12,) + oracle_main_n_grid

oracle_cfg = default_config(
    R=100000,
    R_garch=100000,
    dgps=("garch11_t",),
    methods=(GARCH_ORACLE_METHOD,),
    n_grid=oracle_appendix_n_grid,
    S_grid=oracle_s_grid,
    g_alpha=cfg.g_alpha,
    g_beta=cfg.g_beta,
    garch_dist=cfg.garch_dist,
    nu=cfg.nu,
    burn=500,
    max_workers=1,
)

oracle_cfg


Config(seed=0, alpha=0.05, R=100000, R_garch=100000, dgps=('garch11_t',), methods=('garch11_oracle_analytic',), n_grid=(12, 36, 60, 120, 360, 600), S_grid=(-1.4, -1.2, -1.0, -0.8, -0.6, -0.4, -0.2, 0.0, 0.2, 0.4, 0.6, 0.8, 1.0, 1.2, 1.4), g_alpha=0.05, g_beta=0.9, garch_dist='t', nu=7.0, burn=500, mle_maxiter_warm=80, mle_maxiter_cold=200, mle_tol=1e-06, max_workers=1)

In [8]:
oracle_bundle = run_oracle_bundle(
    oracle_cfg,
    scope="oracle_thesis",
    output_dir=PROJECT_ROOT / "outputs" / "bsc_final",
    main_n_grid=oracle_main_n_grid,
    appendix_n_grid=oracle_appendix_n_grid,
)
oracle_results = oracle_bundle.results
oracle_diagnostics = oracle_bundle.diagnostics
oracle_meta = oracle_bundle.cache_meta
print(f"[oracle] cache_hit={oracle_meta['cache_hit']} hash={oracle_meta['cache_hash']}")


[oracle] cache_hit=True hash=e06fbfad035f


In [9]:
for name, report in oracle_bundle.export_report.items():
    print(
        f"{name}: status={report.status} png={report.png_path} html={report.html_path}"
    )

display(oracle_results)
display(oracle_diagnostics)


oracle_coverage_main: status=png png=/Users/rafaeldecara/Programming/Python/rafsbsc/notebooks/outputs/bsc_final/oracle_coverage_main.png html=None
oracle_se_main: status=png png=/Users/rafaeldecara/Programming/Python/rafsbsc/notebooks/outputs/bsc_final/oracle_se_main.png html=None
oracle_coverage_appendix: status=png png=/Users/rafaeldecara/Programming/Python/rafsbsc/notebooks/outputs/bsc_final/oracle_coverage_appendix.png html=None


,dgp,method,n,S_true,outer_reps,coverage_95,mc_se,mc_lo,mc_hi,reject_rate_H0_S_eq_0,se_cell,omega_hat_cell,se_ratio_meanSE_over_mcSD,fit_fail_count,fit_fail_rate,regularized_count,regularized_rate,fit_converged_rate,fit_attempts_mean,fit_time_ms_mean
0,garch11_t,garch11_oracle_analytic,12,-1.4,100000,0.98569,0.000376,0.984954,0.986426,0.0,1.049904,14.873445,1.653087,0,0.0,0,0.0,NaN,NaN,NaN
1,garch11_t,garch11_oracle_analytic,12,-1.2,100000,0.98357,0.000402,0.982782,0.984358,0.0,0.914401,11.302995,1.613210,0,0.0,0,0.0,NaN,NaN,NaN
2,garch11_t,garch11_oracle_analytic,12,-1.0,100000,0.97956,0.000447,0.978683,0.980437,0.0,0.782311,8.298675,1.549416,0,0.0,0,0.0,NaN,NaN,NaN
3,garch11_t,garch11_oracle_analytic,12,-0.8,100000,0.97518,0.000492,0.974216,0.976144,0.0,0.656468,5.859114,1.461699,0,0.0,0,0.0,NaN,NaN,NaN
4,garch11_t,garch11_oracle_analytic,12,-0.6,100000,0.97365,0.000507,0.972657,0.974643,0.0,0.538156,3.922142,1.359990,0,0.0,0,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,garch11_t,garch11_oracle_analytic,600,0.6,100000,0.95915,0.000626,0.957923,0.960377,1.0,0.066360,2.653189,1.048021,0,0.0,0,0.0,NaN,NaN,NaN
86,garch11_t,garch11_oracle_analytic,600,0.8,100000,0.96059,0.000615,0.959384,0.961796,1.0,0.080801,3.936469,1.059498,0,0.0,0,0.0,NaN,NaN,NaN
87,garch11_t,garch11_oracle_analytic,600,1.0,100000,0.96227,0.000603,0.961089,0.963451,1.0,0.096179,5.579443,1.072293,0,0.0,0,0.0,NaN,NaN,NaN
88,garch11_t,garch11_oracle_analytic,600,1.2,100000,0.96189,0.000605,0.960703,0.963077,1.0,0.112217,7.597860,1.070371,0,0.0,0,0.0,NaN,NaN,NaN


,dgp,n,S_true,bias,rmse,mc_sd_S_hat,outer_reps,garch_outer_reps,garch_fit_fail_count,garch_fit_fail_rate,garch_regularized_count,garch_regularized_rate,garch_fit_converged_rate,garch_fit_attempts_mean,garch_fit_time_ms_mean
0,garch11_t,12,-1.4,-0.245984,0.681086,0.635117,100000,0,0,NaN,0,NaN,NaN,NaN,NaN
1,garch11_t,12,-1.2,-0.210775,0.604739,0.566821,100000,0,0,NaN,0,NaN,NaN,NaN,NaN
2,garch11_t,12,-1.0,-0.175838,0.534647,0.504907,100000,0,0,NaN,0,NaN,NaN,NaN,NaN
3,garch11_t,12,-0.8,-0.142595,0.471205,0.449113,100000,0,0,NaN,0,NaN,NaN,NaN,NaN
4,garch11_t,12,-0.6,-0.106418,0.409764,0.395706,100000,0,0,NaN,0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,garch11_t,600,0.6,0.005721,0.063577,0.063320,100000,0,0,NaN,0,NaN,NaN,NaN,NaN
86,garch11_t,600,0.8,0.008089,0.076691,0.076264,100000,0,0,NaN,0,NaN,NaN,NaN,NaN
87,garch11_t,600,1.0,0.009651,0.090212,0.089695,100000,0,0,NaN,0,NaN,NaN,NaN,NaN
88,garch11_t,600,1.2,0.012146,0.105540,0.104839,100000,0,0,NaN,0,NaN,NaN,NaN,NaN
